# Job Description Crawler — AI / Data Science jobs in Singapore

Crawls ~**1,000 AI/Data-Science job postings** from two sources and normalizes
them into a single JSON schema:

| Source | Method | Notes |
|---|---|---|
| LinkedIn | public *jobs-guest* API (no login) | `sg.indeed.com`-style HTML scraping works here; salary is never shown to guests |
| Indeed SG | [`python-jobspy`](https://github.com/speedyapply/JobSpy) | direct scraping of `sg.indeed.com` is blocked by Cloudflare (HTTP 403); jobspy uses Indeed's internal API instead |

Output schema (one record per job, saved to `output/jobs.json`):

```json
{
  "id": "linkedin_4449083174",
  "job_title": "", "company": "", "location": "", "salary": "",
  "description": "",
  "requirements": [], "responsibilities": [], "benefits": [],
  "employment_type": ""
}
```

`requirements` / `responsibilities` / `benefits` are extracted from the job
description by a heading-based section parser (tested on real scraped JDs:
requirements 34/34, responsibilities 33/34; most JDs genuinely have no
benefits section).

**Crawling is checkpointed** — raw records stream to
`output/checkpoints/*.jsonl` as they are fetched, so if a crawl cell is
interrupted (rate-limit, network, kernel restart) you can simply re-run it and
it resumes where it left off.

In [1]:
%pip install -q python-jobspy requests beautifulsoup4 pandas


[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import random
import re
import time
from itertools import zip_longest
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
from bs4 import BeautifulSoup
from jobspy import scrape_jobs

# ------------------------------------------------------------- search scope
SEARCH_TERMS = [
    "Data Scientist",
    "Machine Learning Engineer",
    "AI Engineer",
    "Data Engineer",
    "Data Analyst",
    "Deep Learning",
    "NLP Engineer",
    "Computer Vision Engineer",
    "MLOps Engineer",
    "Big Data Engineer",
    "Business Intelligence Analyst",
    "AI Research Scientist",
]
LOCATION = "Singapore"

# ----------------------------------------------------------------- targets
TARGET_JOBS = 1000       # final dataset size
LINKEDIN_TARGET = 550    # per-source targets leave headroom: cross-source
INDEED_TARGET = 550      # duplicates are removed before the final trim
INDEED_PER_TERM = 80     # results requested from Indeed per search term

# ------------------------------------------------------------------ output
OUTPUT_DIR = Path("output")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LINKEDIN_CHECKPOINT = CHECKPOINT_DIR / "linkedin_raw.jsonl"
INDEED_CHECKPOINT = CHECKPOINT_DIR / "indeed_raw.jsonl"
JOBS_JSON = OUTPUT_DIR / "jobs.json"

## 1. Job-description section parser

Splits a JD (HTML from either source) into `(heading, bullets)` blocks and
classifies each heading into **requirements** / **responsibilities** /
**benefits** with keyword rules. Handles `<strong>`-style LinkedIn headings,
`<h*>` tags, ALL-CAPS pseudo-headings, `**bold**` markdown, and JDs with no
recognizable headings (those return a clean `description_text` and empty
lists — it never crashes).

In [3]:
"""Job-description section parser.

parse_jd_sections(description, fmt) -> {
    "description_text": str,      # clean plain text of the whole JD
    "requirements": list[str],
    "responsibilities": list[str],
    "benefits": list[str],
}

fmt is "html" or "markdown".

Strategy: linearize the JD into (kind, text) lines where kind is one of
"heading" / "bullet" / "text", then classify headings with keyword rules and
assign the following bullets / text lines to the matched bucket.  Blocks under
unrecognized headings stay only in description_text.
"""

from __future__ import annotations

import re

from bs4 import BeautifulSoup, Tag

# ---------------------------------------------------------------------------
# Sentinels used while flattening HTML to text (stripped from input first).
_B_OPEN = "\x02"    # bold span opens
_B_CLOSE = "\x03"   # bold span closes
_H_OPEN = "\x04"    # <h1>-<h6> span opens
_H_CLOSE = "\x07"   # <h1>-<h6> span closes
_LI_OPEN = "\x05"   # <li> span opens
_LI_CLOSE = "\x06"  # <li> span closes
_SENTINELS = (_B_OPEN, _B_CLOSE, _H_OPEN, _H_CLOSE, _LI_OPEN, _LI_CLOSE)

_BLOCK_TAGS = {
    "p", "div", "section", "article", "ul", "ol", "li", "table", "tr",
    "blockquote", "header", "footer", "h1", "h2", "h3", "h4", "h5", "h6",
}
_HEAD_TAGS = {"h1", "h2", "h3", "h4", "h5", "h6"}

_MAX_HEADING_LEN = 80        # bold/tag headings longer than this are prose
_MAX_PLAIN_HEADING_LEN = 60  # plain-text pseudo-headings must be short

# ---------------------------------------------------------------------------
# Heading classification (case-insensitive, run on normalized heading text).
# Buckets are checked in order: benefits, requirements, responsibilities.

_BENEFIT_PATTERNS = [
    r"benefit", r"perks", r"we offer", r"what's in it for you", r"why join",
    r"compensation", r"\bsalary\b", r"our offer", r"in return",
    r"what you('ll| will)? (get|receive|enjoy)", r"why you'll love",
    r"\brewards?\b", r"why work (with|for|at)", r"remuneration",
    r"what we('ll| will) do for you",
]

_REQUIREMENT_PATTERNS = [
    r"requirement", r"qualification", r"\bskills?\b", r"experience",
    r"who you are", r"what you('ll| will)? bring", r"\byou bring\b",
    r"what (we[' a]re |we )?look(ing)? for", r"must[- ]have", r"about you",
    r"what you need", r"what it takes", r"ideal candidate", r"your profile",
    r"candidate profile", r"competenc", r"nice[- ]to[- ]have",
    r"good[- ]to[- ]have", r"^(preferred|desired|desirable|essential|required)\b",
    r"expertise", r"\bknowledge\b", r"education", r"\babilities\b",
    r"attributes", r"pre-?requisit", r"eligibilit",
    r"who we are looking for", r"what makes you", r"to succeed",
    r"succeed in (this|the) role", r"\bright fit\b",
]

_RESPONSIBILITY_PATTERNS = [
    r"responsibilit", r"duties", r"what you('ll| will) (be )?do",
    r"\bthe role\b", r"your impact", r"day[ -]to[ -]day", r"\byou will\b",
    r"accountabilit", r"your (role|mission|tasks)\b", r"\btasks\b",
    r"scope of", r"role overview", r"in this role", r"job scope",
    r"what will you do", r"what you do", r"key functions", r"job duties",
    r"your contribution", r"how you('ll| will) (make an impact|contribute)",
    r"\bjob purpose\b", r"purpose of the (role|job|position)",
    r"typical work", r"work includes",
]

# "Weak" headings: generic section titles that mean "the duties section" only
# when a bullet list follows them directly (otherwise they head the overview).
_WEAK_RESPONSIBILITY_RE = re.compile(
    r"^(?:description/)?(?:job |position |role |brief )?"
    r"(?:description|summary|overview)(?: \(continued\))?$"
)

# Headings that always end the current section (company boilerplate, legal,
# logistics) even if bullets follow them.
_TERMINATOR_PATTERNS = [
    r"^about\b", r"who we are", r"\bthe (firm|company|team|client)\b",
    r"our (story|purpose|culture|values|commitment|mission|company)",
    r"company (description|overview|profile)", r"equal opportunit", r"\beeo\b",
    r"diversity", r"inclusion", r"how to apply", r"^to apply", r"application",
    r"next steps", r"what's next", r"privacy", r"disclaimer",
    r"other information", r"additional information", r"ea licen[cs]e",
    r"referr", r"location", r"\bcity\b", r"travel", r"relocation",
    r"job (schedule|number|id|posting|segmentation)", r"working hours",
    r"authentic self", r"why work here\b$",
]
_TERMINATOR_RES = [re.compile(p, re.I) for p in _TERMINATOR_PATTERNS]


def _is_terminator(text: str) -> bool:
    norm = _normalize_heading(text)
    return any(p.search(norm) for p in _TERMINATOR_RES)


def _is_weak_responsibility(text: str) -> bool:
    return bool(_WEAK_RESPONSIBILITY_RE.match(_normalize_heading(text)))

_CATEGORY_ORDER = (
    ("benefits", [re.compile(p, re.I) for p in _BENEFIT_PATTERNS]),
    ("requirements", [re.compile(p, re.I) for p in _REQUIREMENT_PATTERNS]),
    ("responsibilities", [re.compile(p, re.I) for p in _RESPONSIBILITY_PATTERNS]),
)

_ALL_PATTERNS = [pat for _, pats in _CATEGORY_ORDER for pat in pats]


def _normalize_heading(text: str) -> str:
    """Lowercase, straighten quotes, collapse whitespace, strip decoration."""
    t = text.replace("’", "'").replace("‘", "'").replace("`", "'")
    t = t.replace("–", "-").replace("—", "-")
    t = t.replace("：", ":").replace("；", ";").replace("。", ".")
    t = re.sub(r"\s+", " ", t).strip()
    t = t.strip(" \t:;-–—*#•.")
    return t.lower()


def classify_heading(text: str) -> str | None:
    """Return 'requirements' / 'responsibilities' / 'benefits' or None."""
    norm = _normalize_heading(text)
    if not norm:
        return None
    for bucket, patterns in _CATEGORY_ORDER:
        for pat in patterns:
            if pat.search(norm):
                return bucket
    return None


def _keyword_match(text: str) -> bool:
    norm = _normalize_heading(text)
    return any(p.search(norm) for p in _ALL_PATTERNS)


# ---------------------------------------------------------------------------
# HTML -> lines

def _lines_from_html(desc: str) -> list[tuple[str, str]]:
    for s in _SENTINELS:
        desc = desc.replace(s, "")
    soup = BeautifulSoup(desc, "html.parser")

    for tag in soup.find_all(["script", "style"]):
        tag.decompose()
    for tag in soup.find_all("br"):
        tag.replace_with("\n")

    # Annotate the tree in place, then flatten once with get_text().
    for tag in soup.find_all(True):
        if not isinstance(tag, Tag):
            continue
        name = tag.name
        if name in _HEAD_TAGS:
            tag.insert(0, _H_OPEN)
            tag.append(_H_CLOSE)
        elif name == "li":
            tag.insert(0, _LI_OPEN)
            tag.append(_LI_CLOSE)
        elif name in ("strong", "b"):
            tag.insert(0, _B_OPEN)
            tag.append(_B_CLOSE)
        if name in _BLOCK_TAGS:
            tag.insert_before("\n")
            tag.append("\n")

    text = soup.get_text()
    return _scan_lines(text)


def _scan_lines(text: str) -> list[tuple[str, str]]:
    """Walk the flattened text, tracking bold/heading/li state across newlines
    (element content often starts with a newline, so per-line marks would land
    on blank lines; depth counters survive the line splits)."""
    lines: list[tuple[str, str]] = []
    bold_depth = head_depth = li_depth = 0
    for raw in text.split("\n"):
        visible_chars = []
        is_head_tag = False
        is_li = False
        all_bold = True
        has_content = False
        for ch in raw:
            if ch == _B_OPEN:
                bold_depth += 1
                continue
            if ch == _B_CLOSE:
                bold_depth = max(0, bold_depth - 1)
                continue
            if ch == _H_OPEN:
                head_depth += 1
                continue
            if ch == _H_CLOSE:
                head_depth = max(0, head_depth - 1)
                continue
            if ch == _LI_OPEN:
                li_depth += 1
                continue
            if ch == _LI_CLOSE:
                li_depth = max(0, li_depth - 1)
                continue
            visible_chars.append(ch)
            if not ch.isspace():
                has_content = True
                if li_depth > 0:
                    is_li = True
                if head_depth > 0:
                    is_head_tag = True
                if bold_depth == 0:
                    all_bold = False
        visible = re.sub(r"\s+", " ", "".join(visible_chars)).strip()
        if not visible:
            lines.append(("blank", ""))
            continue
        if is_li:
            lines.append(("bullet", visible))
        elif is_head_tag and len(visible) <= _MAX_HEADING_LEN:
            lines.append(("heading", visible))
        elif has_content and all_bold and _bold_line_is_heading(visible):
            # Whole-line bold => acts as a heading (LinkedIn <strong> style).
            lines.append(("heading", visible))
        elif _looks_like_text_bullet(visible):
            lines.append(("bullet", _strip_text_bullet(visible)))
        elif _plain_pseudo_heading(visible):
            lines.append(("heading", visible))
        else:
            lines.append(("text", visible))
    return lines


_SENTENCE_END = (".", "!", "?", ";", ",", "。", "；", "，")


def _bold_line_is_heading(line: str) -> bool:
    """A fully-bold line is a heading unless it reads like a sentence/item."""
    if len(line) > _MAX_HEADING_LEN:
        return False
    if line.rstrip().endswith(_SENTENCE_END):
        return False  # bold sentence, e.g. a bolded list item
    if len(line.split()) > 7 and not line.rstrip().endswith((":", "：")):
        return False
    return True


# dash/dot bullets need a space; numeric bullets may omit it ("1.Optimize")
# but must not swallow decimals ("1.5 years").
_TEXT_BULLET_RE = re.compile(r"^\s*(?:[-*•▪●◦·‣–]\s+|\d{1,2}[.)、]\s*(?!\d))")


def _looks_like_text_bullet(line: str) -> bool:
    return bool(_TEXT_BULLET_RE.match(line))


def _strip_text_bullet(line: str) -> str:
    return _TEXT_BULLET_RE.sub("", line).strip()


def _plain_pseudo_heading(line: str) -> bool:
    """Plain (unstyled) line that still reads as a section heading."""
    if len(line) > _MAX_PLAIN_HEADING_LEN:
        return False
    if line.endswith((".", ",", ";", "!", "?", "。", "；", "，")):
        return False
    stripped = line.rstrip(":：").strip()
    if not stripped:
        return False
    letters = [c for c in stripped if c.isalpha()]
    if len(letters) >= 2 and all(c.isupper() for c in letters):
        return True  # ALL-CAPS line
    if line.endswith((":", "：")) and len(stripped.split()) <= 8:
        return True  # short line ending with ':'
    # Short title-ish line matching a category keyword (e.g. "Job Requirements"
    # sitting between blank lines with no styling at all), or a generic section
    # title like "Job Description" whose meaning gets resolved by context.
    words = stripped.split()
    if len(words) <= 6 and stripped[0].isupper() and (
            _keyword_match(stripped) or _is_weak_responsibility(stripped)):
        return True
    return False


# ---------------------------------------------------------------------------
# Markdown -> lines

_MD_HEADING_RE = re.compile(r"^\s{0,3}(#{1,6})\s+(.*?)\s*#*\s*$")
_MD_BOLD_LINE_RE = re.compile(r"^\s*(?:\*\*|__)(.+?)(?:\*\*|__)\s*:?\s*$")


def _strip_md_inline(text: str) -> str:
    text = re.sub(r"(\*\*|__)(.*?)\1", r"\2", text)
    text = re.sub(r"(?<!\w)([*_])([^*_]+)\1(?!\w)", r"\2", text)
    text = re.sub(r"`([^`]*)`", r"\1", text)
    text = re.sub(r"\[([^\]]*)\]\([^)]*\)", r"\1", text)
    return text.strip()


def _lines_from_markdown(desc: str) -> list[tuple[str, str]]:
    lines: list[tuple[str, str]] = []
    for raw in desc.split("\n"):
        line = raw.rstrip()
        if not line.strip():
            lines.append(("blank", ""))
            continue
        m = _MD_HEADING_RE.match(line)
        if m:
            lines.append(("heading", _strip_md_inline(m.group(2))))
            continue
        if _TEXT_BULLET_RE.match(line):
            lines.append(("bullet", _strip_md_inline(_strip_text_bullet(line))))
            continue
        m = _MD_BOLD_LINE_RE.match(line)
        if m and len(m.group(1)) <= _MAX_HEADING_LEN:
            lines.append(("heading", _strip_md_inline(m.group(1))))
            continue
        clean = _strip_md_inline(line)
        if _plain_pseudo_heading(clean):
            lines.append(("heading", clean))
        else:
            lines.append(("text", clean))
    return lines


# ---------------------------------------------------------------------------
# Assemble buckets from the line stream

def _next_is_bullet(lines: list[tuple[str, str]], i: int) -> bool:
    """Is the next content line after index i a bullet (skipping blanks)?"""
    for kind, _ in lines[i + 1:]:
        if kind == "blank":
            continue
        return kind == "bullet"
    return False


def _weak_resolves(lines: list[tuple[str, str]], i: int,
                   text_budget: int = 2) -> bool:
    """Does a weak heading ("Job Description", "Summary", ...) at index i head
    a bullet list?  Allows a short intro (<= text_budget prose lines) and
    unclassified sub-headings before the bullets, but stops at the next
    classified heading."""
    for kind, text in lines[i + 1:]:
        if kind == "blank":
            continue
        if kind == "heading":
            if classify_heading(text):
                return False
            continue
        if kind == "text":
            text_budget -= 1
            if text_budget < 0:
                return False
            continue
        return kind == "bullet"
    return False


def _assemble(lines: list[tuple[str, str]]) -> dict:
    buckets: dict[str, list[str]] = {
        "requirements": [], "responsibilities": [], "benefits": [],
    }
    current: str | None = None
    seen_bullet = False  # bullets already collected under the current heading
    last_heading: str | None = None
    first_classified: int | None = None  # index of first classified heading
    orphan_bullets: list[tuple[int, str | None, str]] = []  # (idx, ctx, text)

    for i, (kind, text) in enumerate(lines):
        if kind == "heading":
            last_heading = text
            bucket = classify_heading(text)
            if bucket:
                current = bucket
                if first_classified is None:
                    first_classified = i
            elif _is_weak_responsibility(text) and _weak_resolves(lines, i):
                # "Job Description" / "Summary" heading a bullet list => duties.
                current = "responsibilities"
                if first_classified is None:
                    first_classified = i
            elif (current and not _is_terminator(text)
                    and _next_is_bullet(lines, i)):
                # Unrecognized sub-heading inside a live section (e.g.
                # "Strategic Leadership" bullets under "Your Role"): keep it.
                pass
            else:
                current = None
            seen_bullet = False
        elif kind == "bullet":
            item = text.strip()
            if not item:
                continue
            if current:
                buckets[current].append(item)
                seen_bullet = True
            else:
                orphan_bullets.append((i, last_heading, item))
        elif kind == "text":
            # Prose lines directly under a classified heading count as items,
            # but once a bullet list has run under this heading, later prose is
            # treated as unrelated trailing copy.
            if current and not seen_bullet:
                item = text.strip()
                if item:
                    buckets[current].append(item)
        # blank lines: no state change

    # Fallback: JDs that list duties under topical sub-headings only (e.g.
    # "Observability Strategy and Governance" ... then "Requirements:").  If no
    # responsibilities section was found, bullets that precede the first
    # classified heading — and are not under boilerplate headings — are duties.
    if not buckets["responsibilities"] and first_classified is not None:
        early = [item for idx, ctx, item in orphan_bullets
                 if idx < first_classified
                 and (ctx is None or not _is_terminator(ctx))]
        if early:
            buckets["responsibilities"] = early

    for key, items in buckets.items():
        deduped, seen = [], set()
        for it in items:
            k = it.lower()
            if k not in seen:
                seen.add(k)
                deduped.append(it)
        buckets[key] = deduped
    return buckets


def _description_text(lines: list[tuple[str, str]]) -> str:
    out: list[str] = []
    prev_blank = True
    for kind, text in lines:
        if kind == "blank" or not text:
            if not prev_blank:
                out.append("")
                prev_blank = True
            continue
        out.append(f"- {text}" if kind == "bullet" else text)
        prev_blank = False
    while out and out[-1] == "":
        out.pop()
    return "\n".join(out)


# ---------------------------------------------------------------------------
# Public API

def parse_jd_sections(description: str, fmt: str) -> dict:
    """Parse a job description into sections.

    description: raw JD (HTML fragment or markdown/plain text).
    fmt: "html" or "markdown".
    """
    empty = {
        "description_text": "",
        "requirements": [],
        "responsibilities": [],
        "benefits": [],
    }
    if not description or not isinstance(description, str):
        return empty
    fmt = (fmt or "").strip().lower()
    try:
        if fmt == "html" or (fmt not in ("markdown", "md", "text", "txt")
                             and re.search(r"<[a-zA-Z][^>]*>", description)):
            lines = _lines_from_html(description)
        else:
            lines = _lines_from_markdown(description)
        result = _assemble(lines)
        result["description_text"] = _description_text(lines)
        return result
    except Exception:
        # Never crash: fall back to a whitespace-normalized plain text dump.
        stripped = re.sub(r"<[^>]+>", " ", description)
        stripped = re.sub(r"[ \t]+", " ", stripped).strip()
        empty["description_text"] = stripped
        return empty


# Convenience: collect every heading the linearizer sees (for tuning/tests).
def extract_headings(description: str, fmt: str) -> list[str]:
    if not description or not isinstance(description, str):
        return []
    fmt = (fmt or "").strip().lower()
    if fmt == "html" or (fmt not in ("markdown", "md", "text", "txt")
                         and re.search(r"<[a-zA-Z][^>]*>", description)):
        lines = _lines_from_html(description)
    else:
        lines = _lines_from_markdown(description)
    return [t for k, t in lines if k == "heading"]

In [4]:
# --------------------------------------------------------- checkpoint helpers
def load_checkpoint(path):
    """Load previously crawled raw records (job_id -> record) from JSONL."""
    records = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            records[rec["job_id"]] = rec
    return records


def append_checkpoint(path, record):
    with open(path, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, ensure_ascii=False) + "\n")


# --------------------------------------------------- salary fallback extractor
# Neither source exposes structured salary for Singapore (verified live:
# 0/12 LinkedIn guest pages, 0/30 Indeed API rows), but some postings embed
# it in the description text, e.g. "S$6,000 - S$8,500 per month".
_CUR = r"(?:S?\$|SGD\s?)"
_AMT = r"\d{1,3}(?:,\d{3})*(?:\.\d+)?\s*[kK]?"
_SALARY_RE = re.compile(
    _CUR + r"\s*" + _AMT
    + r"(?:\s*(?:-|–|—|to|~)\s*" + _CUR + r"?\s*" + _AMT + r")?"
    + r"(?:\s*(?:per|a|an|/)\s*(?:month|mth|mo|annum|year|yr|week|wk|day|hour|hr))?",
    re.IGNORECASE,
)
_INTERVAL_RE = re.compile(
    r"(?:per|a|an|/)\s*(?:month|mth|mo|annum|year|yr|week|wk|day|hour|hr)", re.I)
_RANGE_RE = re.compile(r"(?:-|–|—|\bto\b|~)")


def extract_salary_from_text(text):
    """Best-effort salary mention from JD text. Only accepts matches that
    state a pay interval or a range — a bare "$500" could be anything."""
    if not text:
        return ""
    candidates = [re.sub(r"\s+", " ", m.group(0)).strip()
                  for m in _SALARY_RE.finditer(text)]
    for cand in candidates:
        if _INTERVAL_RE.search(cand):
            return cand
    for cand in candidates:
        if _RANGE_RE.search(cand):
            return cand
    return ""


# ---------------------------------------------------- employment type mapping
# Indeed reports machine values (fulltime, parttime, ...); LinkedIn reports
# display values (Full-time, Contract, ...). Normalize to the LinkedIn style.
_ETYPE_MAP = {
    "fulltime": "Full-time",
    "parttime": "Part-time",
    "contract": "Contract",
    "temporary": "Temporary",
    "internship": "Internship",
    "volunteer": "Volunteer",
    "perdiem": "Per diem",
}


def norm_employment_type(value):
    if not value:
        return ""
    out = []
    for part in re.split(r"[,/]", str(value)):
        part = part.strip()
        if not part:
            continue
        key = re.sub(r"[^a-z]", "", part.lower())
        out.append(_ETYPE_MAP.get(key, part))
    return ", ".join(dict.fromkeys(out))

## 2. LinkedIn crawler (public guest API, no login)

* search: `jobs-guest/jobs/api/seeMoreJobPostings/search?keywords=…&location=…&start=N`
  → HTML fragment of job cards (job id in `data-entity-urn`)
* detail: `jobs-guest/jobs/api/jobPosting/<id>` → title, company, location,
  description HTML (`div.show-more-less-html__markup`) and the job-criteria
  list (seniority / employment type)

Throttling (tuned on live runs): ~1.2–2.4 s jitter between requests, a longer
cool-down every 40 detail fetches, and retry with linear backoff on 429 /
non-200 / empty-body responses (LinkedIn soft-throttles by returning 200 with
an empty body). Every good record is checkpointed immediately; re-running the
crawl cell resumes from the checkpoint.

In [5]:
LI_SEARCH_URL = (
    "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
    "?keywords={kw}&location={loc}&start={start}"
)
LI_DETAIL_URL = "https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/{job_id}"

LI_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

li_session = requests.Session()
li_session.headers.update(LI_HEADERS)


def li_get(url, attempts=5, base_sleep=3.0):
    """GET with polite retry/backoff on 429, non-200, or empty body."""
    for attempt in range(1, attempts + 1):
        try:
            resp = li_session.get(url, timeout=30)
        except requests.RequestException as exc:
            print(f"  [warn] request error ({exc}); retry {attempt}/{attempts}")
            time.sleep(base_sleep * attempt)
            continue
        if resp.status_code == 200 and resp.text.strip():
            return resp
        wait = base_sleep * attempt  # linear backoff: 3, 6, 9, 12 s
        if resp.status_code == 429:
            retry_after = resp.headers.get("Retry-After")
            if retry_after and retry_after.isdigit():
                wait = max(wait, int(retry_after))
        time.sleep(wait)
    return None


def li_text(node):
    return re.sub(r"\s+", " ", node.get_text(" ", strip=True)) if node else ""


def li_extract_job_ids(search_html):
    """Job ids from data-entity-urn attrs, falling back to card hrefs."""
    soup = BeautifulSoup(search_html, "html.parser")
    ids = []
    for card in soup.select("div.base-card[data-entity-urn]"):
        m = re.search(r"(\d+)$", card["data-entity-urn"])
        if m:
            ids.append(m.group(1))
    if not ids:
        for a in soup.select("a.base-card__full-link[href]"):
            m = (re.search(r"-(\d{6,})\?", a["href"])
                 or re.search(r"/jobs/view/[^\"?]*?(\d{6,})", a["href"]))
            if m:
                ids.append(m.group(1))
    return ids


def li_parse_detail(job_id, html):
    soup = BeautifulSoup(html, "html.parser")
    title = li_text(soup.select_one("h1.top-card-layout__title")
                    or soup.select_one(".topcard__title"))
    company = li_text(soup.select_one("a.topcard__org-name-link")
                      or soup.select_one(".topcard__flavor"))
    location = li_text(soup.select_one(".topcard__flavor--bullet"))
    salary = li_text(soup.select_one(".salary")
                     or soup.select_one(".compensation__salary"))
    desc_node = soup.select_one("div.show-more-less-html__markup")
    description_html = desc_node.decode_contents().strip() if desc_node else ""
    criteria = {}
    crit_list = soup.select_one("ul.description__job-criteria-list")
    if crit_list:
        for li in crit_list.select("li"):
            key = li_text(li.select_one(".description__job-criteria-subheader"))
            val = li_text(li.select_one(".description__job-criteria-text"))
            if key:
                criteria[key] = val
    return {
        "source": "linkedin",
        "job_id": job_id,
        "url": f"https://www.linkedin.com/jobs/view/{job_id}",
        "job_title": title,
        "company": company,
        "location": location,
        "salary": salary,
        "employment_type": criteria.get("Employment type", ""),
        "seniority": criteria.get("Seniority level", ""),
        "description_html": description_html,
    }


def li_collect_ids(needed, skip_ids):
    """Paginate the guest search across all terms until `needed` new ids."""
    seen, ordered = set(), []
    for kw in SEARCH_TERMS:
        if len(ordered) >= needed:
            break
        start = 0
        while start <= 900:
            url = LI_SEARCH_URL.format(kw=quote(kw), loc=quote(LOCATION),
                                       start=start)
            resp = li_get(url)
            if resp is None:
                break
            ids = li_extract_job_ids(resp.text)
            if not ids:
                break
            for jid in ids:
                if jid not in seen and jid not in skip_ids:
                    seen.add(jid)
                    ordered.append(jid)
            start += len(ids)
            time.sleep(random.uniform(1.0, 1.8))
            if len(ordered) >= needed:
                break
        print(f"  [{kw}] cumulative new ids: {len(ordered)}")
    return ordered


def crawl_linkedin(target):
    done = load_checkpoint(LINKEDIN_CHECKPOINT)
    print(f"LinkedIn checkpoint: {len(done)} records already crawled")
    if len(done) >= target:
        return list(done.values())

    # 1.4x buffer: some detail fetches fail or come back incomplete
    ids = li_collect_ids(int((target - len(done)) * 1.4) + 10, set(done))
    print(f"fetching details for up to {len(ids)} jobs ...")
    consecutive_failures = 0
    for fetched, jid in enumerate(ids, start=1):
        if len(done) >= target:
            break
        resp = li_get(LI_DETAIL_URL.format(job_id=jid))
        if resp is None:
            consecutive_failures += 1
            if consecutive_failures >= 15:
                print("  aborting: LinkedIn is refusing requests. "
                      "Re-run this cell later — it resumes from the checkpoint.")
                break
            continue
        consecutive_failures = 0
        rec = li_parse_detail(jid, resp.text)
        if rec["job_title"] and rec["company"] and rec["description_html"]:
            done[jid] = rec
            append_checkpoint(LINKEDIN_CHECKPOINT, rec)
            if len(done) % 25 == 0:
                print(f"  {len(done)}/{target} jobs crawled")
        time.sleep(random.uniform(1.2, 2.4))
        if fetched % 40 == 0:
            time.sleep(random.uniform(12, 20))  # periodic cool-down
    print(f"LinkedIn done: {len(done)} records")
    return list(done.values())

In [6]:
linkedin_jobs = crawl_linkedin(LINKEDIN_TARGET)
len(linkedin_jobs)

LinkedIn checkpoint: 550 records already crawled


550

## 3. Indeed Singapore crawler (via `python-jobspy`)

`sg.indeed.com` HTML is behind Cloudflare (HTTP 403 for scripts), so this uses
jobspy's Indeed scraper, which talks to Indeed's internal mobile/GraphQL API.
Descriptions are requested as HTML so the same section parser handles both
sources. Structured salary fields are always null for Singapore (verified),
and ~17% of rows are confidential postings without a company name — those are
dropped. Records are checkpointed per search term.

In [7]:
def _clean(val):
    """Return a stripped string, or None for NaN/None/empty."""
    if val is None:
        return None
    if isinstance(val, float) and pd.isna(val):
        return None
    s = str(val).strip()
    return s or None


def compose_salary(row):
    """Human-readable salary from jobspy's min/max/interval/currency columns."""
    def fmt(x):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return None
        x = float(x)
        return f"{x:,.0f}" if x == int(x) else f"{x:,.2f}"

    lo, hi = fmt(row.get("min_amount")), fmt(row.get("max_amount"))
    if lo is None and hi is None:
        return None
    amount = f"{lo} - {hi}" if (lo and hi and lo != hi) else (lo or hi)
    currency = _clean(row.get("currency")) or ""
    salary = " ".join(p for p in (currency, amount) if p)
    interval = _clean(row.get("interval"))
    return f"{salary} per {interval}" if interval else salary


def crawl_indeed(target):
    done = load_checkpoint(INDEED_CHECKPOINT)
    print(f"Indeed checkpoint: {len(done)} records already crawled")
    for term in SEARCH_TERMS:
        if len(done) >= target:
            break
        print(f"  [indeed] {term!r} ...", end=" ")
        try:
            df = scrape_jobs(
                site_name=["indeed"],
                search_term=term,
                location=LOCATION,
                country_indeed="Singapore",
                results_wanted=INDEED_PER_TERM,
                description_format="html",
                verbose=0,
            )
        except Exception as exc:
            print(f"failed: {exc}")
            time.sleep(5)
            continue
        added = 0
        for _, row in df.iterrows():
            job_id = _clean(row.get("id")) or _clean(row.get("job_url"))
            if not job_id or job_id in done:
                continue
            rec = {
                "source": "indeed",
                "job_id": job_id,
                "url": _clean(row.get("job_url")),
                "job_title": _clean(row.get("title")),
                "company": _clean(row.get("company")),
                "location": _clean(row.get("location")),
                "salary": compose_salary(row),
                "employment_type": _clean(row.get("job_type")),
                "description": _clean(row.get("description")),
            }
            # drop confidential postings / rows without a usable description
            if not (rec["job_title"] and rec["company"] and rec["description"]):
                continue
            done[job_id] = rec
            append_checkpoint(INDEED_CHECKPOINT, rec)
            added += 1
        print(f"{len(df)} rows, +{added} new (total {len(done)})")
        time.sleep(3)  # be polite between queries
    print(f"Indeed done: {len(done)} records")
    return list(done.values())

In [8]:
indeed_jobs = crawl_indeed(INDEED_TARGET)
len(indeed_jobs)

Indeed checkpoint: 555 records already crawled
Indeed done: 555 records


555

## 4. Normalize to the target schema and export

Parses every description into sections, fills salary from the description text
when the sources did not provide one, normalizes employment types, dedupes
(within a source by job id — already done — and across sources by
title + company, keeping the record with the richer description), interleaves
the two sources, trims to `TARGET_JOBS`, and writes `output/jobs.json`.

In [9]:
def normalize(rec):
    desc_raw = rec.get("description_html") or rec.get("description") or ""
    parsed = parse_jd_sections(desc_raw, "html")
    salary = (rec.get("salary")
              or extract_salary_from_text(parsed["description_text"]))
    return {
        "id": f"{rec['source']}_{rec['job_id']}",
        "job_title": rec.get("job_title") or "",
        "company": rec.get("company") or "",
        "location": rec.get("location") or "",
        "salary": salary or "",
        "description": parsed["description_text"],
        "requirements": parsed["requirements"],
        "responsibilities": parsed["responsibilities"],
        "benefits": parsed["benefits"],
        "employment_type": norm_employment_type(rec.get("employment_type")),
    }


def dedupe_across_sources(jobs):
    """Same title+company posted on both sites -> keep the richer record."""
    best, order = {}, []
    for job in jobs:
        key = (job["job_title"].strip().lower(), job["company"].strip().lower())
        if key in best:
            if len(job["description"]) > len(best[key]["description"]):
                best[key] = job
        else:
            best[key] = job
            order.append(key)
    return [best[k] for k in order]


normalized = [normalize(r) for r in linkedin_jobs + indeed_jobs]
deduped = dedupe_across_sources(normalized)

li_final = [j for j in deduped if j["id"].startswith("linkedin_")]
in_final = [j for j in deduped if j["id"].startswith("indeed_")]
mixed = [j for pair in zip_longest(li_final, in_final)
         for j in pair if j is not None]
jobs = mixed[:TARGET_JOBS]

with open(JOBS_JSON, "w", encoding="utf-8") as fh:
    json.dump(jobs, fh, ensure_ascii=False, indent=2)

print(f"crawled     : {len(normalized)} "
      f"(linkedin {len(linkedin_jobs)}, indeed {len(indeed_jobs)})")
print(f"after dedupe: {len(deduped)}")
print(f"saved       : {len(jobs)} -> {JOBS_JSON}")
if len(jobs) < TARGET_JOBS:
    print(f"NOTE: {TARGET_JOBS - len(jobs)} short of target — re-run the two "
          "crawl cells (they resume from checkpoints) and then this cell.")

crawled     : 1105 (linkedin 550, indeed 555)
after dedupe: 1042
saved       : 1000 -> output/jobs.json


In [10]:
# Field coverage + a sample record
n = len(jobs)
print(f"total jobs: {n}\n")
for field in ("salary", "employment_type", "requirements",
              "responsibilities", "benefits"):
    filled = sum(1 for j in jobs if j[field])
    print(f"  {field:18s} non-empty: {filled:4d}/{n}  ({filled / n:4.0%})")

print("\n--- sample record " + "-" * 40)
sample = dict(jobs[0])
sample["description"] = sample["description"][:400] + " ..."
sample["requirements"] = sample["requirements"][:4]
sample["responsibilities"] = sample["responsibilities"][:4]
print(json.dumps(sample, ensure_ascii=False, indent=2))

total jobs: 1000

  salary             non-empty:   24/1000  (  2%)
  employment_type    non-empty:  835/1000  ( 84%)
  requirements       non-empty:  892/1000  ( 89%)
  responsibilities   non-empty:  921/1000  ( 92%)
  benefits           non-empty:  248/1000  ( 25%)

--- sample record ----------------------------------------
{
  "id": "linkedin_4443881898",
  "job_title": "Data Scientist",
  "company": "SAP",
  "location": "Singapore, Singapore",
  "salary": "",
  "description": "We help the world run better\n\nAt SAP, we keep it simple: you bring your best to us, and we'll bring out the best in you. We're builders touching over 20 industries and 80% of global commerce, and we need your unique talents to help shape what's next. The work is challenging – but it matters. You'll find a place where you can be yourself, prioritize your wellbeing, and truly belong. What's in it fo ...",
  "requirements": [
    "Hands-on Data Scientist with 3+ years of experience in machine learning or appli

## 5. Semantic chunking (LangChain `SemanticChunker` + `text-embedding-3-small`)

`output/jobs.json` is kept **unchanged**. For every job record, all fields are
concatenated into one labeled text, which is then split with LangChain's
semantic chunking algorithm using OpenAI `text-embedding-3-small`
(**1536 dimensions** — same embedding setup as `preprocessing_cv.ipynb`).

Each chunk becomes:

```json
{
  "text_chunking": "<one semantic chunk of the concatenated JD text>",
  "metadata": { ...the complete original job record (all fields)... }
}
```

Final output = the untouched original file **plus** `output/jobs_chunks.json`
with all chunk records. Chunking streams to
`output/checkpoints/chunks.jsonl` as it goes, so re-running the cell resumes
instead of re-paying for embeddings (requires `OPENAI_API_KEY` in `.env`).

In [11]:
%pip install -q langchain-experimental langchain-openai python-dotenv


[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [12]:
import warnings
warnings.filterwarnings("ignore", message=".*langchain-experimental.*")

from dotenv import load_dotenv
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

load_dotenv()  # reads .env next to this notebook (OPENAI_API_KEY)

EMBED_MODEL = "text-embedding-3-small"
EMBED_DIMS = 1536

CHUNKS_CHECKPOINT = CHECKPOINT_DIR / "chunks.jsonl"
CHUNKS_JSON = OUTPUT_DIR / "jobs_chunks.json"

embeddings = OpenAIEmbeddings(model=EMBED_MODEL, dimensions=EMBED_DIMS)
# default breakpoint: percentile — split where consecutive sentences'
# embedding distance jumps above the 95th percentile
chunker = SemanticChunker(embeddings)


def job_to_text(job):
    """Concatenate every field of one JD into a single labeled text."""
    def join(value):
        return "; ".join(value) if isinstance(value, list) else (value or "")

    parts = []
    for field in ("job_title", "company", "location", "salary",
                  "employment_type", "description", "requirements",
                  "responsibilities", "benefits"):
        val = join(job[field]).strip()
        if val:
            parts.append(f"{field.replace('_', ' ').title()}: {val}")
    return "\n".join(parts)

In [13]:
jobs = json.loads(JOBS_JSON.read_text(encoding="utf-8"))  # original, untouched
jobs_by_id = {job["id"]: job for job in jobs}

# Resume support: reload chunks already produced in earlier runs.
# Each job's saved chunks are VALIDATED before reuse — if their combined length
# overshoots the job's text, the checkpoint holds more than one chunking of
# that job (this happens if the cell is ever executed by two processes at
# once) and the job is re-chunked from scratch.
# NOTE: never run this cell in two kernels/terminals at the same time.
chunks_by_job = {}
dropped = 0
if CHUNKS_CHECKPOINT.exists():
    loaded = {}
    for line in CHUNKS_CHECKPOINT.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rec = json.loads(line)
            loaded.setdefault(rec["metadata"]["id"], []).append(rec)
    for jid, recs in loaded.items():
        job = jobs_by_id.get(jid)
        total = sum(len(r["text_chunking"]) for r in recs)
        expected = max(len(job_to_text(job)), 1)
        if job is not None and 0.8 * expected <= total <= 1.2 * expected:
            chunks_by_job[jid] = recs
        else:
            dropped += 1
    if dropped:
        # compact the checkpoint down to the validated records only
        with open(CHUNKS_CHECKPOINT, "w", encoding="utf-8") as fh:
            for recs in chunks_by_job.values():
                for rec in recs:
                    fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print(f"dropped {dropped} stale/duplicated jobs from the checkpoint")
print(f"chunk checkpoint: {len(chunks_by_job)}/{len(jobs)} jobs already chunked")

failed = []
for i, job in enumerate(jobs, start=1):
    if job["id"] in chunks_by_job:
        continue
    text = job_to_text(job)
    pieces = None
    for attempt in (1, 2, 3):
        try:
            pieces = chunker.split_text(text)
            break
        except Exception as exc:
            print(f"  [warn] {job['id']} attempt {attempt}: {exc}")
            time.sleep(10 * attempt)
    if pieces is None:
        failed.append(job["id"])
        continue
    records = [{"text_chunking": piece.strip(), "metadata": job}
               for piece in pieces if piece.strip()]
    for rec in records:
        append_checkpoint(CHUNKS_CHECKPOINT, rec)
    chunks_by_job[job["id"]] = records
    if i % 50 == 0:
        print(f"  {i}/{len(jobs)} jobs chunked")

if failed:
    print(f"failed jobs (re-run this cell to retry): {failed}")

# final file, preserving jobs.json order
all_chunks = [rec for job in jobs for rec in chunks_by_job.get(job["id"], [])]
with open(CHUNKS_JSON, "w", encoding="utf-8") as fh:
    json.dump(all_chunks, fh, ensure_ascii=False, indent=2)
print(f"{len(all_chunks)} chunks from {len(chunks_by_job)} jobs -> {CHUNKS_JSON}")
print(f"original file untouched: {JOBS_JSON}")

chunk checkpoint: 1000/1000 jobs already chunked
3017 chunks from 1000 jobs -> output/jobs_chunks.json
original file untouched: output/jobs.json


In [14]:
# chunking stats + one sample chunk record
counts = pd.Series({jid: len(recs) for jid, recs in chunks_by_job.items()})
print(f"jobs chunked : {len(counts)}")
print(f"total chunks : {int(counts.sum())}")
print(f"chunks/job   : mean {counts.mean():.2f}, min {counts.min()}, "
      f"max {counts.max()}")
print(counts.value_counts().sort_index().rename("jobs").to_frame().T)

sample = dict(all_chunks[0])
sample["text_chunking"] = sample["text_chunking"][:300] + " ..."
sample["metadata"] = {
    k: (v[:80] + " ..." if isinstance(v, str) and len(v) > 80 else v)
    for k, v in sample["metadata"].items()
}
print("\n--- sample chunk record " + "-" * 34)
print(json.dumps(sample, ensure_ascii=False, indent=2))

jobs chunked : 1000
total chunks : 3017
chunks/job   : mean 3.02, min 1, max 7
      1    2    3    4   5  6  7
jobs  6  312  422  187  66  6  1

--- sample chunk record ----------------------------------
{
  "text_chunking": "Job Title: Data Scientist\nCompany: SAP\nLocation: Singapore, Singapore\nEmployment Type: Full-time\nDescription: We help the world run better\n\nAt SAP, we keep it simple: you bring your best to us, and we'll bring out the best in you. We're builders touching over 20 industries and 80% of global commerce ...",
  "metadata": {
    "id": "linkedin_4443881898",
    "job_title": "Data Scientist",
    "company": "SAP",
    "location": "Singapore, Singapore",
    "salary": "",
    "description": "We help the world run better\n\nAt SAP, we keep it simple: you bring your best to  ...",
    "requirements": [
      "Hands-on Data Scientist with 3+ years of experience in machine learning or applied AI",
      "A relevant university degree from Data Science, AI or equivalen

## 6. Embeddings → data points

Two kinds of points, both embedded with `text-embedding-3-small` (1536-d),
same conventions as `preprocessing_cv.ipynb` (deterministic `uuid5` point ids
→ idempotent upserts; payload carries the **full original job record** plus
`embedded_text`, the exact string the vector encodes) — all indexed into
**one shared collection**, distinguished by the payload key `type`:

| `type` | source | embedded text | one point per |
|---|---|---|---|
| `chunk` | `output/jobs_chunks.json` | `text_chunking` | semantic chunk |
| `field` | `output/jobs.json` | `description` / `requirements` / `responsibilities` | non-empty field |

All points are also saved to `output/jd_points.jsonl`.

In [15]:
import collections
import uuid

import tiktoken
from openai import (OpenAI, APIError, APIConnectionError, APITimeoutError,
                    InternalServerError, RateLimitError)

ENC = tiktoken.get_encoding("cl100k_base")   # tokenizer of text-embedding-3-*
MAX_TOK = 8000                               # headroom under the 8192 input cap

oai_client = OpenAI()
RETRYABLE = (APIError, APIConnectionError, APITimeoutError, RateLimitError,
             InternalServerError)

chunk_records = json.loads(CHUNKS_JSON.read_text(encoding="utf-8"))
job_records = json.loads(JOBS_JSON.read_text(encoding="utf-8"))

JOB_EMBED_FIELDS = ["description", "requirements", "responsibilities"]


def as_text(value):
    return "; ".join(value) if isinstance(value, list) else (value or "")


def truncate(text):
    """Truncate ONCE so payload's embedded_text is exactly what gets embedded."""
    toks = ENC.encode(text)
    if len(toks) > MAX_TOK:
        return ENC.decode(toks[:MAX_TOK]), MAX_TOK
    return text, len(toks)


# (point_id, payload, text, n_tokens)
items = []

# --- kind 1: one point per semantic chunk (metadata = full job record) -------
chunk_counter = collections.Counter()
for rec in chunk_records:
    job = rec["metadata"]
    idx = chunk_counter[job["id"]]
    chunk_counter[job["id"]] += 1
    text, n_tok = truncate(rec["text_chunking"])
    pid = str(uuid.uuid5(uuid.NAMESPACE_URL, f"jd_chunk:{job['id']}:{idx}"))
    payload = {"type": "chunk", "chunk_index": idx, "embedded_text": text, **job}
    items.append((pid, payload, text, n_tok))

# --- kind 2: one point per non-empty field of each job (same metadata) -------
for job in job_records:
    for field in JOB_EMBED_FIELDS:
        text = as_text(job[field]).strip()
        if not text:
            continue
        text, n_tok = truncate(text)
        pid = str(uuid.uuid5(uuid.NAMESPACE_URL, f"jd_field:{job['id']}:{field}"))
        payload = {"type": "field", "field": field, "embedded_text": text, **job}
        items.append((pid, payload, text, n_tok))

by_type = collections.Counter(p["type"] for _, p, _, _ in items)
print(f"{len(items)} texts to embed ({dict(by_type)}), "
      f"~{sum(n for *_, n in items):,} tokens")

5830 texts to embed ({'chunk': 3017, 'field': 2813}), ~2,540,421 tokens


In [16]:
def embed_batch(texts, attempts=4):
    last_err = None
    for a in range(attempts):
        try:
            resp = oai_client.embeddings.create(model=EMBED_MODEL, input=texts,
                                                dimensions=EMBED_DIMS)
            return [d.embedding for d in sorted(resp.data, key=lambda d: d.index)]
        except RETRYABLE as exc:
            last_err = exc
            if a < attempts - 1:
                time.sleep(2 ** a)
    raise RuntimeError(f"embedding batch failed: {last_err!r}") from last_err


# batch by count AND token budget (API caps: 2048 inputs / 300k tokens/request)
batches, cur, cur_tok = [], [], 0
for *_, text, n in items:
    if cur and (len(cur) == 64 or cur_tok + n > 250_000):
        batches.append(cur)
        cur, cur_tok = [], 0
    cur.append(text)
    cur_tok += n
if cur:
    batches.append(cur)

vectors = []
for b_i, batch in enumerate(batches, start=1):
    vectors.extend(embed_batch(batch))
    if b_i % 10 == 0 or b_i == len(batches):
        print(f"  embedded batch {b_i}/{len(batches)} "
              f"({len(vectors)}/{len(items)} texts)")
assert len(vectors) == len(items), f"{len(vectors)} vectors for {len(items)}"

points = [{"id": pid, "vector": vec, "payload": payload}
          for (pid, payload, _, _), vec in zip(items, vectors, strict=True)]

POINTS_JSONL = OUTPUT_DIR / "jd_points.jsonl"
with open(POINTS_JSONL, "w", encoding="utf-8") as fh:
    for p in points:
        fh.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"saved {len(points)} points ({dict(by_type)}) -> {POINTS_JSONL}")

  embedded batch 10/92 (640/5830 texts)


  embedded batch 20/92 (1280/5830 texts)


  embedded batch 30/92 (1920/5830 texts)


  embedded batch 40/92 (2560/5830 texts)


  embedded batch 50/92 (3200/5830 texts)


...

  embedded batch 70/92 (4480/5830 texts)


  embedded batch 80/92 (5120/5830 texts)


  embedded batch 90/92 (5760/5830 texts)


  embedded batch 92/92 (5830/5830 texts)


saved 5830 points ({'chunk': 3017, 'field': 2813}) -> output/jd_points.jsonl


## 7. Upsert into Qdrant (`http://localhost:6333`)

Same storage stack and auth as the CV pipeline (`QDRANT_API_KEY` from `.env`).
**One collection — `jd_jobs`** (1536-d, Cosine) — holding both point kinds;
filter on the payload key `type` (`chunk` / `field`) to search one kind only.

The collection is dropped and recreated on every run (point counts can shrink
between runs, and recreating guarantees no stale points linger), and point ids
are deterministic `uuid5`. A payload index on the filterable keys speeds up
filtered search, and a live semantic query at the end sanity-checks the data.

In [17]:
import os

QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY", "")
QDRANT_HEADERS = {"api-key": QDRANT_API_KEY} if QDRANT_API_KEY else {}

COLLECTION = "jd_jobs"   # ONE collection for both point kinds (payload "type")

# Recreate on every run: point counts can shrink between runs, and dropping
# the collection first guarantees no stale points linger. The full upsert
# afterwards is only a few seconds of HTTP.
requests.delete(f"{QDRANT_URL}/collections/{COLLECTION}", headers=QDRANT_HEADERS)
r = requests.put(f"{QDRANT_URL}/collections/{COLLECTION}", headers=QDRANT_HEADERS,
                 json={"vectors": {"size": EMBED_DIMS, "distance": "Cosine"}})
if r.status_code != 200:
    raise RuntimeError(f"create failed ({r.status_code}): {r.text[:300]}")
print(f"[{COLLECTION}] collection recreated")

# payload indexes for the keys used to filter searches
for name in ("type", "field", "id", "company", "employment_type"):
    requests.put(f"{QDRANT_URL}/collections/{COLLECTION}/index?wait=true",
                 headers=QDRANT_HEADERS,
                 json={"field_name": name, "field_schema": "keyword"})

for i in range(0, len(points), 100):
    r = requests.put(f"{QDRANT_URL}/collections/{COLLECTION}/points?wait=true",
                     headers=QDRANT_HEADERS,
                     json={"points": points[i:i + 100]})
    if not r.ok:              # surface Qdrant's error body, not just the status
        raise RuntimeError(f"upsert failed ({r.status_code}): {r.text[:300]}")

info = requests.get(f"{QDRANT_URL}/collections/{COLLECTION}",
                    headers=QDRANT_HEADERS).json()
print(f"[{COLLECTION}] points in collection: {info['result']['points_count']}")

[jd_jobs] collection recreated


[jd_jobs] points in collection: 5830


In [18]:
# sanity check: live semantic search on the shared collection
query = "Machine learning engineer with NLP and Python experience"
qvec = embed_batch([query])[0]


def qdrant_search(query_filter=None, limit=3):
    body = {"vector": qvec, "limit": limit,
            "with_payload": ["id", "job_title", "company", "type",
                             "field", "chunk_index"]}
    if query_filter:
        body["filter"] = query_filter
    r = requests.post(f"{QDRANT_URL}/collections/{COLLECTION}/points/search",
                      headers=QDRANT_HEADERS, json=body)
    return r.json()["result"]


def show(hits):
    for hit in hits:
        p = hit["payload"]
        where = p.get("field") or f"chunk {p.get('chunk_index')}"
        print(f"  {hit['score']:.3f}  [{p['type']}] "
              f"{p['job_title']} @ {p['company']} ({where})")


print(f"top 3 overall for: {query!r}")
show(qdrant_search())

for kind in ("chunk", "field"):
    print(f"\ntop 3 with filter type={kind!r}")
    show(qdrant_search({"must": [{"key": "type", "match": {"value": kind}}]}))

top 3 overall for: 'Machine learning engineer with NLP and Python experience'
  0.642  [field] Data Engineer (AI Native, Pretrain Algorithm, data pipeline, production) @ DADACONSULTANTS PTE. LTD. (requirements)
  0.626  [field] AI Research Scientist/Engineer @ Sonar (requirements)
  0.624  [field] AI Engineer @ PERSOL TECH SERVICES PTE. LTD. (requirements)

top 3 with filter type='chunk'
  0.621  [chunk] Senior Data Scientist (Business Transaction Platform) @ Grab (chunk 1)
  0.600  [chunk] Research Assistant (Computational Science and Machine Learning) @ National University of Singapore (chunk 1)
  0.587  [chunk] Geospatial Solutions Engineer @ ST Engineering (chunk 1)

top 3 with filter type='field'
  0.642  [field] Data Engineer (AI Native, Pretrain Algorithm, data pipeline, production) @ DADACONSULTANTS PTE. LTD. (requirements)
  0.626  [field] AI Research Scientist/Engineer @ Sonar (requirements)
  0.624  [field] AI Engineer @ PERSOL TECH SERVICES PTE. LTD. (requirements)
